# Ingest — Azure AI Foundry

Two sources, because cost and usage live in different systems:

| | |
|---|---|
| **Cost Management Query API** | spend and usage quantity, daily, by meter |
| **Azure Monitor Metrics** | token counts and provisioned-capacity utilisation |

Both tables are **optional**. Load neither and the Foundry page is simply empty, which is a
supported state. Spend supplies recorded cost and meter quantities; Monitor adds token/request
counts and PTU utilisation. The shipped cost-per-million measure assumes all selected billing
quantities are in millions of tokens: it is not valid for mixed units or provisioned hours.

**Source conventions to check in your tenant** (original observations from 2026-08-07):

- The shipped Foundry cost measure filters specifically to **`Foundry Models`**. Legacy/other
  service names can be collected but are not included in that card. No currency conversion is performed.
- The Cost Management dimension is **`Meter`**, not `MeterName`. The API rejects `MeterName` outright
  and lists the valid dimensions in the 400 body.
- Azure Monitor metric names have changed: **`InputTokens`**, **`OutputTokens`**,
  **`ProvisionedUtilization`** — older material says `ProcessedPromptTokens`, `GeneratedTokens` and
  `AzureOpenAIProvisionedManagedUtilizationV2`. An `OpenAI`-kind resource may still publish the old
  names and an `AIServices`-kind one the new, so this notebook asks the resource which it has.
  Missing or null telemetry is not proof of inactivity; keep real zeros, not fabricated ones.
  Revised collectors require tenant acceptance testing; offline checks do not prove live refresh.

**Copilot Studio pay-as-you-go appears in this data too**, as `Pay As You Go Copilot Credit` at
the rate observed in the original sample. Compare like-for-like periods, currencies and scopes;
differences can reflect billing latency, discounts or adjustments as well as rate assumptions.

**Copilot pay-as-you-go also gets its own table**, `copilot_payg_spend`. Power Platform billing
policies charge Copilot Studio and Cowork credits to the Azure subscription linked to each policy,
which is not always `SUBSCRIPTION_ID`. Section 2 reads `SUBSCRIPTION_ID` plus any subscriptions in
`PAYG_SUBSCRIPTION_IDS`, and splits Studio from Cowork by the `serviceName` tag Azure puts on the
charges. The application needs Cost Management Reader on each of those subscriptions.

Authentication is explicit: an Entra application credential is read from Key Vault and exchanged
for an ARM token. Fabric's documented `getToken` audiences do **not** include ARM; do not assume
`getToken("https://management.azure.com/")` or an automatic notebook managed identity works.
The notebook execution identity needs Key Vault secret-read and Lakehouse write access; the
application needs Azure cost, resource-discovery and metrics read permissions. See the
[Fabric setup guide](https://github.com/microsoft/ConsumptionCentral-for-Microsoft-Copilot/blob/main/docs/ADVANCED-SETUP.md#azure-ingestion-in-fabric) for identities and scheduling.

Each successful run replaces the last `DAYS` complete UTC days, including empty results.
Sections 1-3 collect and validate everything before their three Delta tables are written; any
failure there stops the run. Sections 4-6 then add the Azure capacity, solution spend and billing
reconciliation tables (`azure_deployment_health`, `azure_solution_spend`,
`azure_billing_reconciliation`). They are best effort: an account, deployment or metric call that
fails is logged as a `WARNING` and skipped, and a table whose source fails entirely is left as it
was. Refresh the semantic model only after the notebook finishes; the Delta writes are not one
transaction.


## Configure

In [ ]:
SUBSCRIPTION_ID = "00000000-0000-0000-0000-000000000000"
TENANT_ID = "00000000-0000-0000-0000-000000000000"
CLIENT_ID = "00000000-0000-0000-0000-000000000000"
KEY_VAULT_URL = "https://<your-vault>.vault.azure.net/"
CLIENT_SECRET_NAME = "consumption-central-azure-client-secret"

# Days of history. Cost Management keeps far more; 90 is enough to see a trend
# without making the first run slow.
DAYS = 90

# Resource-tag keys to try, in order, for department attribution. Tenants name
# this differently and none of them being present is fine - DepartmentTag is
# optional everywhere it is used.
TAG_KEYS = ["Department", "department", "CostCentre", "CostCenter",
            "Team", "BusinessUnit", "Owner"]

# Extra subscriptions that Power Platform billing policies charge Copilot pay-as-you-go
# to. SUBSCRIPTION_ID is always read; list others here, for example
# ["11111111-1111-1111-1111-111111111111"]. Leave empty when there are none.
PAYG_SUBSCRIPTION_IDS = []

LAKEHOUSE_SPEND  = "azure_ai_spend"
LAKEHOUSE_TOKENS = "azure_ai_tokens"
LAKEHOUSE_PAYG   = "copilot_payg_spend"
LAKEHOUSE_HEALTH   = "azure_deployment_health"
LAKEHOUSE_SOLUTION = "azure_solution_spend"
LAKEHOUSE_RECON    = "azure_billing_reconciliation"


In [ ]:
import requests
import math
import time
from copy import deepcopy
from datetime import date, datetime, timedelta, timezone
from email.utils import parsedate_to_datetime
from urllib.parse import urlsplit
from uuid import UUID

ARM   = "https://management.azure.com"
_token = None
_token_until = 0


def arm_headers():
    global _token, _token_until
    if _token is None or time.monotonic() >= _token_until:
        secret = notebookutils.credentials.getSecret(KEY_VAULT_URL, CLIENT_SECRET_NAME)
        r = requests.post(
            f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token",
            data={"grant_type": "client_credentials", "client_id": CLIENT_ID,
                  "client_secret": secret, "scope": f"{ARM}/.default"},
            timeout=90, allow_redirects=False)
        r.raise_for_status()
        payload = r.json()
        _token = payload["access_token"]
        _token_until = time.monotonic() + max(0, int(payload["expires_in"]) - 300)
    return {"Authorization": f"Bearer {_token}", "Content-Type": "application/json"}


def retry_delay(headers, attempt):
    delays = [float(2 ** attempt)]
    for name, value in headers.items():
        key = name.lower()
        if key == "retry-after" or (key.startswith("x-ms-ratelimit-") and key.endswith("retry-after")):
            try:
                delay = float(value)
            except ValueError:
                if key != "retry-after":
                    raise ValueError("Invalid Cost Management retry header") from None
                retry_at = parsedate_to_datetime(value)
                if retry_at.tzinfo is None:
                    retry_at = retry_at.replace(tzinfo=timezone.utc)
                delay = (retry_at - datetime.now(timezone.utc)).total_seconds()
            if not math.isfinite(delay):
                raise ValueError("Invalid retry delay")
            delays.append(max(0, delay))
    delay = max(delays)
    if delay > 300:
        raise RuntimeError("Server retry delay exceeds 300 seconds; defer the job")
    return delay


def arm_request(method, url, **kwargs):
    # Continuation URLs are opaque, but must never receive credentials off ARM.
    parsed = urlsplit(url)
    if (parsed.scheme != "https" or parsed.netloc.lower() != "management.azure.com"
            or parsed.fragment):
        raise ValueError("Invalid ARM request or continuation URL")
    for attempt in range(5):
        r = requests.request(method, url, headers=arm_headers(), timeout=180,
                             allow_redirects=False, **kwargs)
        if r.status_code in (429, 502, 503, 504) and attempt < 4:
            delay = retry_delay(r.headers, attempt)
            print(f"ARM HTTP {r.status_code}; retrying the same request in {delay:g}s")
            r.close()
            time.sleep(delay)
            continue
        r.raise_for_status()
        if r.status_code not in (200, 204):
            raise RuntimeError(f"Unexpected ARM status {r.status_code}")
        return r
    raise RuntimeError("ARM retry budget exhausted")


def arm_get(url, **params):
    return arm_request("GET", url, params=params).json()


def arm_list(url, **params):
    items, seen = [], set()
    while url:
        if url in seen:
            raise RuntimeError("Repeated ARM continuation URL")
        seen.add(url)
        page = arm_get(url, **params)
        items.extend(page["value"])
        url, params = page.get("nextLink"), {}
    return items


def cost_query(url, body, required=None):
    records, seen = [], set()
    while url:
        if url in seen:
            raise RuntimeError("Repeated Cost Management continuation URL")
        seen.add(url)
        response = arm_request("POST", url, json=body)
        if response.status_code == 204:
            if records or len(seen) > 1:
                raise RuntimeError("Unexpected empty Cost Management continuation page")
            return []
        page = response.json()["properties"]
        columns = [c["name"] for c in page["columns"]]
        if len(columns) != len(set(columns)):
            raise ValueError("Duplicate cost columns")
        for values in page["rows"]:
            if len(values) != len(columns):
                raise ValueError("Cost row does not match its page columns")
            row = dict(zip(columns, values))
            for alias, spec in body["dataset"]["aggregation"].items():
                candidates = [spec["name"], alias]
                if spec["name"] == "Cost":
                    candidates.append("PreTaxCost")
                found = [key for key in candidates if key in row]
                if not found:
                    raise ValueError(f"Missing cost aggregation {spec['name']}")
                row[spec["name"]] = finite_number(row[found[0]])
            for key in (required or [g["name"] for g in body["dataset"]["grouping"]]) + ["UsageDate", "Currency"]:
                if key not in row:
                    raise ValueError(f"Missing cost column {key}")
            records.append(row)
        url = page.get("nextLink")
    return records


def finite_number(value):
    number = float(value)
    if not math.isfinite(number):
        raise ValueError("Non-finite numeric value in Azure response")
    return number


def resource_group(resource_id):
    parts = resource_id.split("/")
    lower = [p.lower() for p in parts]
    return parts[lower.index("resourcegroups") + 1] if "resourcegroups" in lower else ""


## 1. Spend, from Cost Management

In [ ]:
for identifier in (SUBSCRIPTION_ID, TENANT_ID, CLIENT_ID):
    if UUID(identifier).int == 0:
        raise ValueError("Configure subscription, tenant and application client IDs before running")
if not isinstance(DAYS, int) or isinstance(DAYS, bool) or not 1 <= DAYS <= 90:
    raise ValueError("DAYS must be an integer from 1 to 90 (Monitor retention)")

end   = datetime.now(timezone.utc).date()
start = end - timedelta(days=DAYS)

body = {
    "type": "ActualCost",
    "timeframe": "Custom",
    "timePeriod": {"from": f"{start}T00:00:00Z",
                   "to": f"{end - timedelta(days=1)}T23:59:59Z"},
    "dataset": {
        "granularity": "Daily",
        "aggregation": {
            "totalCost": {"name": "Cost", "function": "Sum"},
            "totalQty":  {"name": "UsageQuantity", "function": "Sum"},
        },
        # The documented limit is two grouping clauses. Discover service/category
        # pairs first, then query Meter x ResourceId within each pair.
        "grouping": [{"type": "Dimension", "name": d} for d in
                     ["ServiceName", "MeterCategory"]],
        "filter": {"dimensions": {
            "name": "ServiceName",
            "operator": "In",
            # 'Foundry Models' is the real one. The others are included so a
            # tenant on older SKUs still returns something.
            "values": ["Foundry Models", "Microsoft Copilot Studio",
                       "Cognitive Services", "Azure Machine Learning",
                       "Azure OpenAI"],
        }},
    },
}

cost_url = (f"{ARM}/subscriptions/{SUBSCRIPTION_ID}/providers/Microsoft.CostManagement"
            "/query?api-version=2025-03-01")
summary = cost_query(cost_url, body)
pairs = {(r["ServiceName"], r["MeterCategory"]) for r in summary}
rows = []
for service, category in sorted(pairs):
    if not service or not category:
        raise ValueError("Cost query returned a blank service/category; cannot safely partition")
    detail = deepcopy(body)
    detail["dataset"]["grouping"] = [
        {"type": "Dimension", "name": d} for d in ["Meter", "ResourceId"]]
    detail["dataset"]["filter"] = {"and": [
        {"dimensions": {"name": key, "operator": "In", "values": [value]}}
        for key, value in [("ServiceName", service), ("MeterCategory", category)]]}
    details = cost_query(cost_url, detail)
    if not details:
        raise RuntimeError("Cost detail is empty for a discovered service/category; rerun")
    for row in details:
        row.update(ServiceName=service, MeterCategory=category)
    rows.extend(details)
print(f"{len(rows):,} cost rows")


In [ ]:
# Resource tags, fetched separately and joined on resource id. Grouping the
# cost query on a TagKey would be tidier but changes a query that is verified
# working.
tags = {}
res = arm_list(f"{ARM}/subscriptions/{SUBSCRIPTION_ID}/resources",
              **{"api-version": "2021-04-01"})
for r in res:
    t = r.get("tags") or {}
    for k in TAG_KEYS:
        if k in t:
            tags[r["id"].lower()] = t[k]
            break
print(f"{len(tags):,} resources carry a department tag")


In [ ]:
recs = []
cost_keys = set()
for r in rows:
    rid   = str(r["ResourceId"] or "")
    parts = rid.split("/")
    usage_date = datetime.strptime(str(r["UsageDate"]), "%Y%m%d").date()
    if not start <= usage_date < end:
        raise ValueError("Cost date outside the requested UTC window")
    key = (usage_date, r["ServiceName"], r["MeterCategory"], r["Meter"],
           rid.lower(), r["Currency"])
    if key in cost_keys:
        raise ValueError("Duplicate aggregated cost grain; refusing to double count")
    cost_keys.add(key)
    recs.append({
        "UsageDate":     usage_date,
        "ServiceName":   r["ServiceName"],
        "MeterCategory": r["MeterCategory"],
        "Meter":         r["Meter"],
        "ResourceId":    rid,
        "ResourceName":  parts[-1] if parts else "",
        "ResourceGroup": resource_group(rid),
        "Cost":          r["Cost"],
        "UsageQuantity": r["UsageQuantity"],
        "Currency":      r["Currency"],
        "DepartmentTag": tags.get(rid.lower(), ""),
    })

print(f"validated {len(recs):,} spend rows; writes wait for metrics collection")


## 2. Copilot pay-as-you-go, billed in Azure

Copilot Studio and Cowork credits beyond prepaid capacity are billed through Power Platform
billing policies, each linked to an Azure subscription. Cost Management reports them under the
service `Microsoft Copilot Studio`, as the meter `Pay As You Go Copilot Credit` at $0.01 a credit
when this was written. Azure tags Cowork charges `serviceName: cowork`; Copilot Studio charges
carry no tag. Grain is day x subscription x meter x tag; `UsageQuantity` is credits only for the
credit meter. Billing can lag usage by a day or more, and no currency conversion is performed.

In [ ]:
# The same two-clause grouping the probe verified: Meter x the serviceName tag,
# filtered to the Copilot Studio service. One query per subscription.
PAYG_SERVICE = "Microsoft Copilot Studio"
PAYG_PRODUCTS = {"": "Copilot Studio", "cowork": "Cowork"}

if not isinstance(PAYG_SUBSCRIPTION_IDS, (list, tuple)):
    raise ValueError("PAYG_SUBSCRIPTION_IDS must be a list of subscription IDs")
payg_subscriptions = []
for identifier in [SUBSCRIPTION_ID, *PAYG_SUBSCRIPTION_IDS]:
    subscription = str(UUID(str(identifier)))
    if UUID(subscription).int == 0:
        raise ValueError("Configure real subscription IDs in PAYG_SUBSCRIPTION_IDS")
    if subscription not in payg_subscriptions:
        payg_subscriptions.append(subscription)

payg_query = deepcopy(body)
payg_query["dataset"]["grouping"] = [{"type": "Dimension", "name": "Meter"},
                                     {"type": "TagKey", "name": "serviceName"}]
payg_query["dataset"]["filter"] = {"dimensions": {
    "name": "ServiceName", "operator": "In", "values": [PAYG_SERVICE]}}

payg_recs, payg_keys = [], set()
for subscription in payg_subscriptions:
    url = (f"{ARM}/subscriptions/{subscription}/providers/Microsoft.CostManagement"
           "/query?api-version=2025-03-01")
    found = cost_query(url, payg_query, required=["Meter", "TagKey", "TagValue"])
    for r in found:
        tag_key = str(r["TagKey"] or "")
        if tag_key.lower() not in ("", "servicename"):
            raise ValueError(f"Unexpected cost tag key {tag_key!r}; refusing to guess the product")
        tag = str(r["TagValue"] or "") if tag_key else ""
        meter = str(r["Meter"] or "")
        if not meter:
            raise ValueError("Copilot cost row has no meter")
        usage_date = datetime.strptime(str(r["UsageDate"]), "%Y%m%d").date()
        if not start <= usage_date < end:
            raise ValueError("Cost date outside the requested UTC window")
        key = (usage_date, subscription, meter, tag, r["Currency"])
        if key in payg_keys:
            raise ValueError("Duplicate Copilot cost grain; refusing to double count")
        payg_keys.add(key)
        payg_recs.append({
            "UsageDate":      usage_date,
            "SubscriptionId": subscription,
            "Meter":          meter,
            "ServiceTag":     tag,
            "Product":        PAYG_PRODUCTS.get(tag.lower(), tag),
            "Cost":           r["Cost"],
            "UsageQuantity":  r["UsageQuantity"],
            "Currency":       r["Currency"],
        })
    print(f"  {subscription}: {len(found):,} Copilot pay-as-you-go rows")

print(f"validated {len(payg_recs):,} Copilot pay-as-you-go rows; writes wait for metrics collection")

## 3. Tokens and provisioned utilisation, from Azure Monitor

Run this section even without provisioned deployments: token/request metrics can still exist.
Counters use `Total`; utilisation uses server-side daily `Average`. Only deployment is split,
with other dimensions rolled up by Azure, so averages are never summed or averaged again.
One name per equivalent metric family is selected; `TotalTokens` is a fallback only when
neither input nor output metrics exists. Do not sum all `Metric` values together.
Zero points are retained; absent/null points mean missing telemetry, not proven zero traffic.

In [ ]:
METRIC_FAMILIES = [
    ("InputTokens", "ProcessedPromptTokens"),
    ("OutputTokens", "GeneratedTokens"),
    ("ModelRequests", "AzureOpenAIRequests", "TotalCalls"),
    ("ProvisionedUtilization", "AzureOpenAIProvisionedManagedUtilizationV2"),
]
UTILIZATION = set(METRIC_FAMILIES[-1])
SERIES_LIMIT = 10000


def select_metrics(definitions, kind):
    available = {d["name"]["value"]: d for d in definitions}
    selected = []
    for family in METRIC_FAMILIES:
        name = next((n for n in family if n in available
                     and not (n == "TotalCalls" and kind.lower() == "openai")), None)
        if name:
            selected.append(name)
    if not any(n in selected for family in METRIC_FAMILIES[:2] for n in family):
        if "TotalTokens" in available:
            selected.append("TotalTokens")
    return [(name, available[name]) for name in selected]


def metric_parameters(name, definition, first, last):
    aggregation = "Average" if name in UTILIZATION else "Total"
    supported = definition.get("supportedAggregationTypes") or [
        definition["primaryAggregationType"]]
    if aggregation.lower() not in {s.lower() for s in supported}:
        raise ValueError(f"{name} does not support required {aggregation} aggregation")
    dimensions = [d["value"] for d in definition.get("dimensions", [])]
    deployment = next((d for candidate in ("modeldeploymentname", "deploymentname", "deployment")
                       for d in dimensions if d.lower() == candidate), None)
    params = {"api-version": "2023-10-01", "metricnames": name,
              "aggregation": aggregation, "interval": "P1D",
              "timespan": f"{first}T00:00:00Z/{last}T00:00:00Z",
              "AutoAdjustTimegrain": "false", "ValidateDimensions": "true"}
    if definition.get("namespace"):
        params["metricnamespace"] = definition["namespace"]
    if deployment:
        params.update({"$filter": f"{deployment} eq '*'", "top": SERIES_LIMIT})
    return params, deployment, aggregation.lower()


def metric_points(payload, name, deployment_dimension, field, account, first, last):
    if payload.get("error"):
        raise RuntimeError(f"Azure Monitor error: {payload['error']}")
    metrics = payload["value"]
    if len(metrics) != 1 or metrics[0]["name"]["value"] != name:
        raise ValueError(f"Azure Monitor omitted or changed requested metric {name}")
    metric = metrics[0]
    if metric.get("errorCode") not in (None, "Success", "0"):
        raise RuntimeError(f"{name}: {metric['errorCode']}: {metric.get('errorMessage', '')}")
    if payload.get("interval") != "P1D":
        raise ValueError(f"{name}: expected daily points; refusing to reaggregate averages")
    series = metric["timeseries"]
    if len(series) >= SERIES_LIMIT:
        raise RuntimeError(f"{name}: series limit reached; results may be truncated")
    result, seen_series, seen_points = [], set(), set()
    for series_item in series:
        metadata = {m["name"]["value"].lower(): m["value"]
                    for m in series_item.get("metadatavalues", [])}
        expected = deployment_dimension.lower() if deployment_dimension else None
        if any(key != expected and value not in ("", None) for key, value in metadata.items()):
            raise ValueError(f"{name}: unexpected dimension split; refusing to double count")
        if expected and expected not in metadata:
            raise ValueError(f"{name}: missing deployment metadata")
        deployment = metadata.get(expected, "") or ""
        if deployment == "*" or deployment in seen_series:
            raise ValueError(f"{name}: duplicate or rolled-up deployment series")
        seen_series.add(deployment)
        for point in series_item["data"]:
            stamp = datetime.fromisoformat(point["timeStamp"].replace("Z", "+00:00"))
            if stamp.tzinfo is None:
                raise ValueError("Metric timestamp is missing its timezone")
            stamp = stamp.astimezone(timezone.utc)
            day = stamp.date()
            # Some APIs include the upper endpoint; adjacent windows must not overlap.
            if day == last and stamp.hour == stamp.minute == stamp.second == stamp.microsecond == 0:
                continue
            if not first <= day < last or (stamp.hour, stamp.minute, stamp.second, stamp.microsecond) != (0, 0, 0, 0):
                raise ValueError(f"{name}: unexpected daily timestamp")
            if field not in point and any(k in point for k in ("total", "average", "count", "minimum", "maximum")):
                raise ValueError(f"{name}: missing requested {field} aggregation")
            value = point.get(field)
            if value is None:
                continue
            key = (day, deployment)
            if key in seen_points:
                raise ValueError(f"{name}: duplicate daily metric point")
            seen_points.add(key)
            result.append({"Date": day, "ResourceName": account["name"],
                           "ResourceGroup": resource_group(account["id"]),
                           "Deployment": deployment, "Metric": name,
                           "Value": finite_number(value)})
    return result


accounts = arm_list(
    f"{ARM}/subscriptions/{SUBSCRIPTION_ID}/providers"
    f"/Microsoft.CognitiveServices/accounts",
    **{"api-version": "2023-05-01"})

points = []
account_ids = set()
for a in accounts:
    if a["id"].lower() in account_ids:
        raise ValueError("Duplicate Cognitive Services account in paginated inventory")
    account_ids.add(a["id"].lower())
    defs = arm_list(f"{ARM}{a['id']}/providers/microsoft.insights/metricDefinitions",
                    **{"api-version": "2018-01-01"})
    ask = select_metrics(defs, a.get("kind", ""))
    if not ask:
        print(f"  {a['name']}: publishes none of the known token metrics")
        continue
    print(f"  {a['name']}: {', '.join(name for name, _ in ask)}")
    for name, definition in ask:
        first = start
        while first < end:
            last = min(first + timedelta(days=30), end)
            params, dimension, field = metric_parameters(name, definition, first, last)
            md = arm_get(f"{ARM}{a['id']}/providers/microsoft.insights/metrics", **params)
            points.extend(metric_points(md, name, dimension, field, a, first, last))
            first = last

SPEND_SCHEMA = ("UsageDate date, ServiceName string, MeterCategory string, Meter string, "
                "ResourceId string, ResourceName string, ResourceGroup string, Cost double, "
                "UsageQuantity double, Currency string, DepartmentTag string")
TOKENS_SCHEMA = ("Date date, ResourceName string, ResourceGroup string, Deployment string, "
                 "Metric string, Value double")
PAYG_SCHEMA = ("UsageDate date, SubscriptionId string, Meter string, ServiceTag string, "
               "Product string, Cost double, UsageQuantity double, Currency string")
spend_frame = spark.createDataFrame(recs, schema=SPEND_SCHEMA)
tokens_frame = spark.createDataFrame(points, schema=TOKENS_SCHEMA)
payg_frame = spark.createDataFrame(payg_recs, schema=PAYG_SCHEMA)
for frame, table, count in [(spend_frame, LAKEHOUSE_SPEND, len(recs)),
                            (tokens_frame, LAKEHOUSE_TOKENS, len(points)),
                            (payg_frame, LAKEHOUSE_PAYG, len(payg_recs))]:
    frame.write.mode("overwrite").format("delta").saveAsTable(table)
    print(f"wrote {table}: {count:,} rows")
if not points:
    print("No metric points returned. Missing telemetry is not proof of zero traffic.")


## 4. Capacity, solution spend and reconciliation: shared helpers

Pure functions with no I/O. The CSV/SharePoint collector (`pull_azure_ai.py`) carries an
identical copy; a repository test fails if the two drift apart.

In [ ]:
import re


HEALTH_HEADERS = ["SnapshotDate", "MetricDate", "SubscriptionName", "SubscriptionId",
                  "ResourceId", "DeploymentId", "DeploymentName", "Application", "ModelName",
                  "ModelVersion", "SkuName", "Region", "PtuCapacity", "MeanUtilizationPct",
                  "PeakUtilizationPct", "Requests", "ServerErrors5xx", "Throttles429",
                  "MeanLatencyMs"]
RECON_HEADERS = ["Period", "PeriodStart", "PeriodEnd", "Product", "PoolName", "Representation",
                 "Amount", "Currency", "Status", "EvidenceType", "Notes"]
SOLUTION_HEADERS = ["UsageDate", "SubscriptionName", "SubscriptionId", "ResourceId",
                    "ResourceName", "ResourceGroup", "Application", "DepartmentTag",
                    "ServiceName", "ServiceCategory", "ActualCost", "AmortizedCost", "Currency",
                    "TotalTokensM", "PaygTokensM", "InputPaygCost", "OutputPaygCost",
                    "CachedPaygCost", "Requests", "SpeechHours", "DocumentPages", "Images",
                    "AllocationStatus", "PricingModel", "BillingScope"]
APPLICATION_TAG_KEYS = ["Application", "App", "Solution", "Workload", "Project"]
HEALTH_REQUEST_METRICS = ("ModelRequests", "AzureOpenAIRequests")
HEALTH_UTILIZATION_METRICS = ("ProvisionedUtilization", "AzureOpenAIProvisionedManagedUtilizationV2")
HEALTH_LATENCY_METRICS = ("TimeToLastByte", "AzureOpenAITTLTInMS",
                          "TimeToResponse", "AzureOpenAITimeToResponse")
TOKEN_METRICS = {"inputtokens", "processedprompttokens", "outputtokens", "generatedtokens",
                 "totaltokens"}
REQUEST_METRICS = {"modelrequests", "azureopenairequests", "totalcalls"}
# Cost Management service names. Copilot Studio pay-as-you-go is deliberately excluded:
# it has its own table and is not Foundry spend.
SOLUTION_AI_SERVICES = ["Foundry Models", "Azure OpenAI", "Cognitive Services", "Foundry Tools",
                        "Azure Machine Learning", "Azure AI Search", "Azure Cognitive Search"]
MODEL_SERVICES = {"foundry models", "azure openai"}
PAYG_POOL = "Foundry PAYG Pool"
PROVISIONED_POOL = "Foundry Provisioned Pool"
RECON_TOLERANCES = ((0.01, "Reconciled"), (0.05, "Variance within tolerance"))


def as_number(value):
    """Finite float, or None for anything missing or invalid. Never raises."""
    if value is None or isinstance(value, bool):
        return None
    try:
        number = float(value)
    except (TypeError, ValueError, OverflowError):
        return None
    return number if math.isfinite(number) else None


def as_day(value):
    if isinstance(value, datetime):
        return value.date()
    if isinstance(value, date):
        return value
    raw = str(value or "")
    for pattern in ("%Y%m%d", "%Y-%m-%d"):
        try:
            return datetime.strptime(raw[:10] if "-" in raw else raw, pattern).date()
        except ValueError:
            continue
    return None


def tag_value(tags, keys):
    lowered = {str(k).lower(): v for k, v in (tags or {}).items()}
    for key in keys:
        value = lowered.get(key.lower())
        if value not in (None, ""):
            return str(value)
    return ""


def month_end(day):
    following = (day.replace(day=28) + timedelta(days=4)).replace(day=1)
    return following - timedelta(days=1)


def metric_windows(start, end, days=30):
    windows, first = [], start
    while first < end:
        last = min(first + timedelta(days=days), end)
        windows.append((first, last))
        first = last
    return windows


def is_provisioned(text):
    lowered = str(text or "").lower()
    return "provisioned" in lowered or re.search(r"\bptu\b", lowered) is not None


def deployment_inventory(account, deployments, warnings):
    """One dict per deployment, keyed by lower-case account resource ID."""
    rid = str(account.get("id") or "").lower()
    application = tag_value(account.get("tags"), APPLICATION_TAG_KEYS)
    found = []
    for deployment in deployments or []:
        try:
            name = str(deployment["name"])
            props = deployment.get("properties") or {}
            model = props.get("model") or {}
            sku = deployment.get("sku") or {}
            sku_name = str(sku.get("name") or "")
            found.append({
                "ResourceId": rid,
                "DeploymentName": name,
                "Application": tag_value(deployment.get("tags"), APPLICATION_TAG_KEYS) or application,
                "ModelName": str(model.get("name") or ""),
                "ModelVersion": str(model.get("version") or ""),
                "SkuName": sku_name,
                "Region": str(account.get("location") or ""),
                # Standard/GlobalStandard capacity is a rate limit (K TPM), not PTUs.
                "PtuCapacity": as_number(sku.get("capacity")) if is_provisioned(sku_name) else None,
            })
        except (KeyError, TypeError, AttributeError) as exc:
            warnings.append(f"{rid}: skipped malformed deployment ({type(exc).__name__})")
    return found


def _dimension(definition, names):
    for item in definition.get("dimensions") or []:
        value = item.get("value") if isinstance(item, dict) else None
        if value and value.lower() in names:
            return value
    return None


def _aggregations(definition):
    supported = definition.get("supportedAggregationTypes") or [
        definition.get("primaryAggregationType")]
    return {str(s).lower() for s in supported if s}


def health_metric_requests(definitions):
    """Plan batched Monitor calls: one for requests by status, one for utilisation + latency."""
    available = {}
    for definition in definitions or []:
        try:
            available.setdefault(definition["name"]["value"], definition)
        except (KeyError, TypeError):
            continue
    deployment_names = ("modeldeploymentname", "deploymentname", "deployment")

    def pick(family, needed):
        for name in family:
            definition = available.get(name)
            if not definition or not needed <= _aggregations(definition):
                continue
            dimension = _dimension(definition, deployment_names)
            if dimension:
                return name, definition, dimension
        return None

    plans = []
    found = pick(HEALTH_REQUEST_METRICS, {"total"})
    if found:
        name, definition, dimension = found
        status = _dimension(definition, ("statuscode",))
        plans.append({
            "metricnames": [name], "aggregation": "Total",
            "filter": f"{dimension} eq '*'" + (f" and {status} eq '*'" if status else ""),
            "roles": {name: "requests"}, "deployment": dimension, "status": status,
            "namespace": definition.get("namespace")})
    grouped = {}
    for family, role in ((HEALTH_UTILIZATION_METRICS, "utilization"),
                         (HEALTH_LATENCY_METRICS, "latency")):
        found = pick(family, {"average"})
        if not found:
            continue
        name, definition, dimension = found
        plan = grouped.setdefault((dimension, definition.get("namespace")), {
            "metricnames": [], "maximum": True, "filter": f"{dimension} eq '*'", "roles": {},
            "deployment": dimension, "status": None, "namespace": definition.get("namespace")})
        plan["metricnames"].append(name)
        plan["roles"][name] = role
        plan["maximum"] = plan["maximum"] and "maximum" in _aggregations(definition)
    for plan in grouped.values():
        plan["aggregation"] = "Average,Maximum" if plan.pop("maximum") else "Average"
        plans.append(plan)
    return plans


def health_metric_parameters(plan, names, first, last):
    params = {"api-version": "2023-10-01", "metricnames": ",".join(names),
              "aggregation": plan["aggregation"], "interval": "P1D",
              "timespan": f"{first}T00:00:00Z/{last}T00:00:00Z",
              "AutoAdjustTimegrain": "false", "$filter": plan["filter"], "top": 10000}
    if plan.get("namespace"):
        params["metricnamespace"] = plan["namespace"]
    return params


def health_accumulate(acc, payload, plan, resource_id, first, last, warnings, label):
    """Fold one Monitor response into acc[(resource, deployment, day)]. Skips bad data."""
    rid = str(resource_id).lower()
    deployment_key = plan["deployment"].lower()
    status_key = (plan.get("status") or "").lower()
    for metric in (payload or {}).get("value") or []:
        try:
            name = metric["name"]["value"]
        except (KeyError, TypeError):
            continue
        role = plan["roles"].get(name)
        if role is None:
            continue
        if metric.get("errorCode") not in (None, "Success", "0", 0):
            warnings.append(f"{label}: {name} returned {metric.get('errorCode')}; skipped")
            continue
        for series in metric.get("timeseries") or []:
            meta = {}
            for item in series.get("metadatavalues") or []:
                try:
                    meta[str(item["name"]["value"]).lower()] = item.get("value")
                except (KeyError, TypeError):
                    continue
            deployment = str(meta.get(deployment_key) or "")
            if not deployment or deployment == "*":
                continue
            status = str(meta.get(status_key) or "") if status_key else ""
            for point in series.get("data") or []:
                try:
                    stamp = datetime.fromisoformat(str(point["timeStamp"]).replace("Z", "+00:00"))
                except (KeyError, TypeError, ValueError):
                    continue
                if stamp.tzinfo is not None:
                    stamp = stamp.astimezone(timezone.utc)
                day = stamp.date()
                if not first <= day < last:
                    continue
                slot = acc.setdefault((rid, deployment.lower(), day), {"name": deployment})
                if role == "requests":
                    value = as_number(point.get("total"))
                    if value is None:
                        continue
                    slot["requests"] = slot.get("requests", 0.0) + value
                    if status_key:
                        # The status split proves the zeroes; without it they stay blank.
                        slot["throttles"] = slot.get("throttles", 0.0) + (value if status == "429" else 0.0)
                        slot["errors"] = slot.get("errors", 0.0) + (value if status.startswith("5") else 0.0)
                elif role == "utilization":
                    average, maximum = as_number(point.get("average")), as_number(point.get("maximum"))
                    if average is not None:
                        slot["utilization"] = average
                    if maximum is not None:
                        slot["peak"] = maximum
                elif role == "latency":
                    average = as_number(point.get("average"))
                    if average is not None:
                        slot["latency"] = average


def health_rows(inventory, acc, start, end, subscription_id, subscription_name):
    """One row per deployment per calendar month (MetricDate = month end).

    Past months list deployments that reported telemetry; the current month lists every
    inventoried deployment so blanks read as "No telemetry" instead of disappearing.
    """
    last_day = end - timedelta(days=1)
    current = month_end(last_day)
    months = {}
    for (rid, deployment, day), slot in acc.items():
        if start <= day < end:
            months.setdefault((rid, deployment, month_end(day)), []).append(slot)
    known = {}
    for item in inventory:
        known.setdefault((item["ResourceId"].lower(), item["DeploymentName"].lower()), item)
    keys = set(months) | {(rid, deployment, current) for rid, deployment in known}
    rows = []
    for rid, deployment, metric_date in sorted(keys):
        slots = months.get((rid, deployment, metric_date), [])
        info = known.get((rid, deployment)) or {
            "ResourceId": rid, "DeploymentName": slots[0]["name"] if slots else deployment,
            "Application": "", "ModelName": "", "ModelVersion": "", "SkuName": "", "Region": "",
            "PtuCapacity": None}
        provisioned = is_provisioned(info["SkuName"])

        def total(field):
            values = [s[field] for s in slots if field in s]
            return round(sum(values), 4) if values else None

        utilization = [s["utilization"] for s in slots if "utilization" in s] if provisioned else []
        peaks = [s["peak"] for s in slots if "peak" in s] if provisioned else []
        latency = [(s["latency"], s.get("requests") or 0.0) for s in slots if "latency" in s]
        weight = sum(w for _, w in latency)
        if latency and weight > 0:
            mean_latency = sum(v * w for v, w in latency) / weight
        elif latency:
            mean_latency = sum(v for v, _ in latency) / len(latency)
        else:
            mean_latency = None
        rows.append(dict(zip(HEALTH_HEADERS, [
            min(metric_date, last_day), metric_date, subscription_name, subscription_id, rid,
            info["DeploymentName"], info["DeploymentName"], info["Application"],
            info["ModelName"], info["ModelVersion"], info["SkuName"], info["Region"],
            info["PtuCapacity"],
            round(sum(utilization) / len(utilization), 2) if utilization else None,
            round(max(peaks), 2) if peaks else None,
            total("requests"), total("errors"), total("throttles"),
            round(mean_latency, 2) if mean_latency is not None else None,
        ])))
    return rows


def token_meter(meter):
    """(kind, millions of tokens per billed unit) for a token meter, else None."""
    text = str(meter or "").lower()
    match = re.search(r"(?:\b(\d+(?:\.\d+)?)\s*([km])\s+)?tokens?\b", text)
    if not match:
        return None
    # The retail price sheet bills token meters that name no unit per 1K tokens.
    size, unit = match.group(1) or "1", match.group(2) or "k"
    millions = float(size) * (0.001 if unit == "k" else 1.0)
    words = set(re.findall(r"[a-z0-9]+", text))
    if words & {"cchd", "cached", "cache", "cd"}:
        kind = "cached"
    elif words & {"outp", "output", "opt", "outpt", "out"}:
        kind = "output"
    else:
        kind = "input"
    return kind, millions


def service_category(service):
    lowered = str(service or "").lower()
    if lowered in MODEL_SERVICES:
        return "Model"
    if lowered in {s.lower() for s in SOLUTION_AI_SERVICES}:
        return "AI service"
    return "Supporting"


def monitor_usage(points, accounts):
    """(day, account id) -> tokens (millions) and requests, from the Monitor token table."""
    ids = {(str(a.get("name") or "").lower(), resource_group(str(a.get("id") or "")).lower()):
           str(a.get("id") or "").lower() for a in accounts}
    usage = {}
    for point in points:
        rid = ids.get((str(point.get("ResourceName") or "").lower(),
                       str(point.get("ResourceGroup") or "").lower()))
        day, value = as_day(point.get("Date")), as_number(point.get("Value"))
        metric = str(point.get("Metric") or "").lower()
        # Monitor returns a zero for every quiet day; only real usage gets a row.
        if not rid or day is None or not value:
            continue
        if metric in TOKEN_METRICS:
            key, value = "tokens", value / 1e6
        elif metric in REQUEST_METRICS:
            key = "requests"
        else:
            continue
        slot = usage.setdefault((day, rid), {})
        slot[key] = slot.get(key, 0.0) + value
    return usage


def solution_rows(actual, amortized, meters, usage, resources, subscription_id,
                  subscription_name, warnings):
    """Daily AI-solution spend by resource and service, plus PAYG token detail and usage."""
    rows, names = {}, {}
    for field, source in (("ActualCost", actual), ("AmortizedCost", amortized)):
        for record in source:
            day, cost = as_day(record.get("UsageDate")), as_number(record.get("Cost"))
            resource_id = str(record.get("ResourceId") or "")
            currency = str(record.get("Currency") or "").strip()
            if day is None or cost is None or not resource_id or not currency:
                continue
            rid = resource_id.lower()
            names.setdefault(rid, resource_id)
            key = (day, rid, str(record.get("ServiceName") or ""), currency)
            row = rows.setdefault(key, {"ActualCost": 0.0, "AmortizedCost": 0.0})
            row[field] += cost
    detail = {}
    for record in meters:
        service = str(record.get("ServiceName") or "")
        day, cost = as_day(record.get("UsageDate")), as_number(record.get("Cost"))
        resource_id = str(record.get("ResourceId") or "")
        if day is None or cost is None or not resource_id or service_category(service) != "Model":
            continue
        key = (day, resource_id.lower(), service, str(record.get("Currency") or "").strip())
        slot = detail.setdefault(key, {"payg": False, "provisioned": False})
        token = token_meter(record.get("Meter"))
        if token:
            kind, millions = token
            slot["payg"] = True
            slot[kind] = slot.get(kind, 0.0) + cost
            quantity = as_number(record.get("UsageQuantity"))
            if quantity is not None:
                slot["tokens"] = slot.get("tokens", 0.0) + quantity * millions
        elif is_provisioned(record.get("Meter")):
            slot["provisioned"] = True
    currencies = {key[3] for key in rows}
    # Attach Monitor usage to exactly one row per resource-day, or the sums double count.
    owners = {}
    for key in sorted(rows, key=lambda k: (service_category(k[2]) != "Model", k)):
        owners.setdefault((key[0], key[1]), key)
    for day_rid, values in usage.items():
        if day_rid in owners or not values:
            continue
        if len(currencies) != 1:
            warnings.append("Monitor usage without matching cost rows skipped: currency unknown")
            continue
        key = (day_rid[0], day_rid[1], "Foundry Models", next(iter(currencies)))
        rows[key] = {"ActualCost": 0.0, "AmortizedCost": 0.0}
        owners[day_rid] = key
    owner_of = {key: day_rid for day_rid, key in owners.items()}
    result = []
    for key in sorted(rows):
        day, rid, service, currency = key
        row, resource_id = rows[key], names.get(rid, rid)
        info = resources.get(rid) or {}
        application = tag_value(info.get("tags"), APPLICATION_TAG_KEYS)
        department = str(info.get("department") or "")
        meter = detail.get(key)
        used = usage.get(owner_of[key], {}) if key in owner_of else {}
        if meter and meter["payg"] and meter["provisioned"]:
            pricing = "PAYG + Provisioned"
        elif meter and meter["provisioned"]:
            pricing = "Provisioned"
        else:
            pricing = "PAYG"

        def money(field):
            value = meter.get(field) if meter else None
            return round(value, 6) if value is not None else None

        result.append(dict(zip(SOLUTION_HEADERS, [
            day, subscription_name, subscription_id, rid, resource_id.split("/")[-1],
            resource_group(resource_id), application, department, service,
            service_category(service), round(row["ActualCost"], 6), round(row["AmortizedCost"], 6),
            currency,
            round(used["tokens"], 6) if "tokens" in used else None,
            round(meter["tokens"], 6) if meter and "tokens" in meter else None,
            money("input"), money("output"), money("cached"),
            used.get("requests"), None, None, None,
            "Tagged" if application or department else "Untagged", pricing,
            f"Subscription: {subscription_name or subscription_id}",
        ])))
    return result


def retail_price_map(items):
    """meterId -> list-price details, preferring the base tier in the primary region."""
    best = {}
    for item in items:
        if str(item.get("type") or "") != "Consumption":
            continue
        meter_id = str(item.get("meterId") or "").lower()
        price = as_number(item.get("retailPrice"))
        if not meter_id or price is None:
            continue
        rank = ((as_number(item.get("tierMinimumUnits")) or 0.0) != 0.0,
                not item.get("isPrimaryMeterRegion", True))
        if meter_id not in best or rank < best[meter_id][0]:
            best[meter_id] = (rank, {
                "price": price, "unit": str(item.get("unitOfMeasure") or ""),
                "meterName": str(item.get("meterName") or ""),
                "productName": str(item.get("productName") or ""),
                "currency": str(item.get("currencyCode") or "")})
    return {meter_id: value for meter_id, (_, value) in best.items()}


def recon_rows(costs, prices, start, end):
    """Monthly UsageEstimate (metered quantity x retail list price) vs AzureActual."""
    last_day = end - timedelta(days=1)
    groups = {}
    for record in costs:
        day, cost = as_day(record.get("UsageDate")), as_number(record.get("Cost"))
        currency = str(record.get("Currency") or "").strip()
        if day is None or cost is None or not currency or not start <= day < end:
            continue
        meter_id = str(record.get("ResourceGuid") or "").lower()
        price = prices.get(meter_id)
        if price and price["currency"] and price["currency"].upper() != currency.upper():
            price = None
        pool = (PROVISIONED_POOL if price and is_provisioned(
            price["meterName"] + " " + price["productName"]) else PAYG_POOL)
        key = (month_end(day), str(record.get("ServiceName") or ""), pool, currency)
        group = groups.setdefault(key, {"actual": 0.0, "estimate": 0.0, "priced": set(),
                                        "unpriced": set(), "unpriced_cost": 0.0})
        group["actual"] += cost
        quantity = as_number(record.get("UsageQuantity"))
        if price and quantity is not None:
            # Cost Management quantities are in the meter's unit of measure, as is the price.
            group["estimate"] += quantity * price["price"]
            group["priced"].add(meter_id)
        else:
            group["unpriced"].add(meter_id or "(no meter id)")
            group["unpriced_cost"] += cost
    rows = []
    for (period_end, product, pool, currency), group in sorted(groups.items()):
        actual, estimate = round(group["actual"], 2), round(group["estimate"], 2)
        if not group["priced"]:
            status, estimate = "Unpriced", None
        else:
            # Judge only the priced meters; unpriced cost is called out in Notes instead.
            priced_actual = group["actual"] - group["unpriced_cost"]
            variance = abs(group["estimate"] - priced_actual) / max(abs(priced_actual), 0.01)
            status = next((label for limit, label in RECON_TOLERANCES if variance <= limit),
                          "Unexplained variance")
        note = f"Retail list price x metered quantity for {len(group['priced'])} meter(s)."
        if group["unpriced"]:
            note += (f" {len(group['unpriced'])} meter(s) had no retail price"
                     f" ({group['unpriced_cost']:.2f} {currency} of actual cost not estimated).")
        note += " List prices exclude negotiated discounts, credits and reservations."
        period = [period_end.strftime("%Y-%m"), max(period_end.replace(day=1), start),
                  min(period_end, last_day), product, pool]
        rows.append(dict(zip(RECON_HEADERS, period + [
            "UsageEstimate", estimate, currency, status, "Retail Prices API x usage quantity", note])))
        rows.append(dict(zip(RECON_HEADERS, period + [
            "AzureActual", actual, currency, status, "Cost Management query",
            "ActualCost from the Cost Management Query API for the period."])))
    return rows


## 5. Deployment health, from ARM and Azure Monitor

One row per deployment per calendar month (`MetricDate` = month end) for the
**Azure: Capacity & Health** page. ARM supplies the inventory (model, version, SKU, region,
capacity); Monitor supplies daily requests split by status code (429 throttles, 5xx errors),
provisioned utilisation and latency. Requests are one batched call per 30-day window;
utilisation and latency share a second call. A failed batch is retried one metric at a time,
then skipped with a warning.

`PtuCapacity` and utilisation are only populated for provisioned SKUs: for Standard and
GlobalStandard, `capacity` is a rate limit, not PTUs, and there is no utilisation metric.
Utilisation is the mean of daily averages (0-100); latency is weighted by daily requests.
The current month lists every deployment, so an idle one shows as no telemetry rather than
disappearing.

In [ ]:
detail_warnings = []


def note(message):
    detail_warnings.append(message)
    print(f"WARNING: {message}")


def flush(found):
    for message in found:
        note(message)
    found.clear()


try:
    subscription_name = str(arm_get(f"{ARM}/subscriptions/{SUBSCRIPTION_ID}",
                                    **{"api-version": "2022-12-01"}).get("displayName") or "")
except Exception as exc:
    subscription_name = ""
    note(f"subscription name unavailable ({type(exc).__name__}: {exc})")

inventory, health_acc, found = [], {}, []
inventory_failures = metric_calls = metric_failures = 0
for a in accounts:
    try:
        deployments = arm_list(f"{ARM}{a['id']}/deployments", **{"api-version": "2023-05-01"})
    except Exception as exc:
        inventory_failures += 1
        note(f"{a['name']}: deployment inventory failed ({type(exc).__name__}: {exc})")
        continue
    inventory.extend(deployment_inventory(a, deployments, found))
    flush(found)
    if not deployments:
        continue
    try:
        definitions = arm_list(f"{ARM}{a['id']}/providers/microsoft.insights/metricDefinitions",
                               **{"api-version": "2018-01-01"})
    except Exception as exc:
        note(f"{a['name']}: metric definitions unavailable ({type(exc).__name__}: {exc})")
        continue
    plans = health_metric_requests(definitions)
    if not plans:
        note(f"{a['name']}: no per-deployment request, utilisation or latency metrics advertised")
    for plan in plans:
        for first, last in metric_windows(start, end):
            # One batched call per plan and window; if it fails, retry its metrics one by one.
            batches = [plan["metricnames"]]
            while batches:
                names = batches.pop()
                metric_calls += 1
                try:
                    payload = arm_get(f"{ARM}{a['id']}/providers/microsoft.insights/metrics",
                                      **health_metric_parameters(plan, names, first, last))
                    health_accumulate(health_acc, payload, plan, a["id"], first, last, found, a["name"])
                except Exception as exc:
                    metric_failures += 1
                    note(f"{a['name']}: {','.join(names)} {first}..{last} failed "
                         f"({type(exc).__name__}: {exc})")
                    if len(names) > 1:
                        batches.extend([name] for name in names)
                flush(found)

health = health_rows(inventory, health_acc, start, end, SUBSCRIPTION_ID, subscription_name)
HEALTH_SCHEMA = ("SnapshotDate date, MetricDate date, SubscriptionName string, SubscriptionId string, "
                 "ResourceId string, DeploymentId string, DeploymentName string, Application string, "
                 "ModelName string, ModelVersion string, SkuName string, Region string, "
                 "PtuCapacity double, MeanUtilizationPct double, PeakUtilizationPct double, "
                 "Requests double, ServerErrors5xx double, Throttles429 double, MeanLatencyMs double")
print(f"{len(inventory):,} deployments; {metric_calls - metric_failures:,}/{metric_calls:,} metric calls succeeded")
if accounts and inventory_failures == len(accounts):
    note(f"every deployment inventory call failed; {LAKEHOUSE_HEALTH} left unchanged")
else:
    spark.createDataFrame(health, schema=HEALTH_SCHEMA).write.mode("overwrite") \
        .format("delta").saveAsTable(LAKEHOUSE_HEALTH)
    print(f"wrote {LAKEHOUSE_HEALTH}: {len(health):,} rows")

## 6. Solution spend and billing reconciliation, from Cost Management

**`azure_solution_spend`** (Foundry spend page): daily ActualCost and AmortizedCost by resource
and service, for AI services plus any other resource in a resource group that holds an AI
account (`Supporting`). PAYG token cost and tokens come from the token meters collected in
section 1; total tokens and requests come from the Monitor points in section 3, attached to one
row per resource and day. Speech hours, document pages and images are not collected yet and
stay blank.

**`azure_billing_reconciliation`** (Billing: Reconciliation page): per month, service and pool,
`UsageEstimate` (metered quantity x the public Azure Retail Prices list price for each meter)
against `AzureActual` (Cost Management ActualCost). List prices exclude negotiated discounts,
credits and reservations, so variance is expected under an enterprise agreement; meters without
a list price are reported as `Unpriced` with their cost in `Notes`. The Retail Prices API is
public and is called without credentials.

In [ ]:
RETAIL_PRICES = "https://prices.azure.com/api/retail/prices"
RETAIL_BATCH = 15


def cost_detail(cost_type, grouping, cost_filter):
    query = deepcopy(body)
    query["type"] = cost_type
    query["dataset"]["grouping"] = [{"type": "Dimension", "name": g} for g in grouping]
    query["dataset"]["filter"] = cost_filter
    return cost_query(cost_url, query)


def retail_prices(meter_ids, currency):
    # Public, unauthenticated API: no ARM token is ever sent here.
    if not re.fullmatch(r"[A-Z]{3}", currency):
        raise ValueError("Unexpected currency code")
    url = RETAIL_PRICES
    params = {"currencyCode": currency,
              "$filter": " or ".join(f"meterId eq '{UUID(m)}'" for m in meter_ids)}
    items, seen = [], set()
    while url:
        parsed = urlsplit(url)
        if (parsed.scheme != "https" or parsed.netloc.lower() != "prices.azure.com"
                or url in seen or len(seen) >= 50):
            raise ValueError("Unexpected Retail Prices continuation URL")
        seen.add(url)
        for attempt in range(4):
            r = requests.get(url, params=params, timeout=60, allow_redirects=False)
            if r.status_code in (429, 500, 502, 503, 504) and attempt < 3:
                r.close()
                time.sleep(2 ** (attempt + 1))
                continue
            break
        r.raise_for_status()
        page = r.json()
        items.extend(page.get("Items") or [])
        url, params = page.get("NextPageLink"), None
    return items


service_filter = {"dimensions": {"name": "ServiceName", "operator": "In",
                                 "values": SOLUTION_AI_SERVICES}}
ai_groups = sorted({resource_group(a["id"]) for a in accounts} - {""})
# Supporting services are the non-AI resources that share a resource group with an AI account.
solution_filter = ({"or": [service_filter, {"dimensions": {
    "name": "ResourceGroupName", "operator": "In", "values": ai_groups}}]}
    if ai_groups else service_filter)
SOLUTION_SCHEMA = ("UsageDate date, SubscriptionName string, SubscriptionId string, ResourceId string, "
                   "ResourceName string, ResourceGroup string, Application string, DepartmentTag string, "
                   "ServiceName string, ServiceCategory string, ActualCost double, AmortizedCost double, "
                   "Currency string, TotalTokensM double, PaygTokensM double, InputPaygCost double, "
                   "OutputPaygCost double, CachedPaygCost double, Requests double, SpeechHours double, "
                   "DocumentPages double, Images double, AllocationStatus string, PricingModel string, "
                   "BillingScope string")
RECON_SCHEMA = ("Period string, PeriodStart date, PeriodEnd date, Product string, PoolName string, "
                "Representation string, Amount double, Currency string, Status string, "
                "EvidenceType string, Notes string")

try:
    actual_rows = cost_detail("ActualCost", ["ResourceId", "ServiceName"], solution_filter)
    amortized_rows = cost_detail("AmortizedCost", ["ResourceId", "ServiceName"], solution_filter)
    resources = {r["id"].lower(): {"tags": r.get("tags"), "department": tags.get(r["id"].lower(), "")}
                 for r in res}
    solution = solution_rows(actual_rows, amortized_rows, recs, monitor_usage(points, accounts),
                             resources, SUBSCRIPTION_ID, subscription_name, found)
    flush(found)
    spark.createDataFrame(solution, schema=SOLUTION_SCHEMA).write.mode("overwrite") \
        .format("delta").saveAsTable(LAKEHOUSE_SOLUTION)
    print(f"wrote {LAKEHOUSE_SOLUTION}: {len(solution):,} rows")
except Exception as exc:
    note(f"solution spend failed; {LAKEHOUSE_SOLUTION} left unchanged ({type(exc).__name__}: {exc})")

try:
    meter_costs = cost_detail("ActualCost", ["ResourceGuid", "ServiceName"], service_filter)
    prices = {}
    for currency in sorted({str(r["Currency"]).upper() for r in meter_costs}):
        ids = []
        for r in meter_costs:
            try:
                if str(r["Currency"]).upper() == currency:
                    ids.append(str(UUID(str(r["ResourceGuid"]))))
            except ValueError:
                continue
        ids = sorted(set(ids))
        for i in range(0, len(ids), RETAIL_BATCH):
            try:
                prices.update(retail_price_map(retail_prices(ids[i:i + RETAIL_BATCH], currency)))
            except Exception as exc:
                note(f"retail prices unavailable for {len(ids[i:i + RETAIL_BATCH])} {currency} meters "
                     f"({type(exc).__name__}: {exc}); they are reported as unpriced")
    recon = recon_rows(meter_costs, prices, start, end)
    spark.createDataFrame(recon, schema=RECON_SCHEMA).write.mode("overwrite") \
        .format("delta").saveAsTable(LAKEHOUSE_RECON)
    print(f"wrote {LAKEHOUSE_RECON}: {len(recon):,} rows")
except Exception as exc:
    note(f"billing reconciliation failed; {LAKEHOUSE_RECON} left unchanged ({type(exc).__name__}: {exc})")

if detail_warnings:
    print(f"{len(detail_warnings):,} warning(s) in sections 5-6; the tables above were still written "
          "where their source succeeded.")